# CSA vs old A/C0/C — saved captions, CPU only
Accelerator **NONE**, Internet **ON**, enable Secret `KAGGLE_API_TOKEN` (or change SECRET_NAME to `codex`). **No Add Input or checkpoint.**
Downloads only the small review ZIP from exact notebook139 and historical caption text. Does not train/generate captions/load models.
Common4000: re-score A,C0,C,CSA9,CSA10 with identical first5references/image and identical PTB/CIDEr/CHAIR cache.
Full5000: score CSA9/10 separately with original25010references. Output CIDEr/BLEU4/ROUGE-L/CHAIRs/i/recall/false counts/length.
Historical A/C0/C metrics and stored CSA5000 CIDEr must replay; otherwise stop. Best9 fixed before CHAIR.
Loss: includes CSA NLL history; old comparable NLL absent, vocabularies differ. No fabricated cross-model loss ranking.
Oldgreedy vs CSAbeam5 and historically reused validation remain limitations; this is not an independent test.
V2 restores exact old image-ID/input order and original CSA5000rank-concat order. Saves actual/expected metric diagnostics.
All CHAIR outputs are preserved even if CIDEr replay fails; any replay mismatch still makes overall comparison INCOMPLETE.
Send `csa_saved_quality_v139_v2_review.zip`. CPU dependency/scorer execution remains user-run pending.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,re,shutil
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/139'
SECRET_NAME='KAGGLE_API_TOKEN' # Change to codex if that is your actual Secret name.
OUTPUT=Path('/kaggle/working/csa_saved_quality_v139_v2')
COMMIT='3a6532941a98384a8501e7dcfb5be3453ec407ab'
HASHES={'score_saved_quality.py': '19bced5082a59d51c933413a40662ecd0fbf5e79031f408ea4b8fad39e043bf5', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'comparison_baseline4000.json.gz': '8360e82244995afb12fc4eb0ede17d4a092d7f7b846105c7fcd59cb15cc7e046', 'COMPARISON_INPUTS.json': '68ad3fdf4af3ac4f38e0125ceb8d87e2085bdf02e7d3e60d9ca41bcc321dce69', 'QUALITY_COMPARISON_PROTOCOL.md': '7ef4fe5b3f000822ce89b46bee774e937a30533d7863dddd28de5a823c7e6ceb'}
if OUTPUT.exists():raise RuntimeError('Preserve existing output and use a fresh session/path.')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-saved-quality-'));LOG=OUTPUT/'notebook.log'
for n,h in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{n}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Changed source:'+n)
    (SOURCE/n).write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/139':raise ValueError('This notebook pins139; do not substitute latest or another version.')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/139', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v139', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v139; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CPU metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('CPU metric cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    DOWNLOAD=SOURCE/'download';DOWNLOAD.mkdir()
    run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),
             '--file-pattern',r'(^|/)csa_full_xe_10_v2_review\.zip$'])
    manifest=json.loads((SOURCE/'COMPARISON_INPUTS.json').read_text(encoding='utf-8'))
    matches=[p for p in DOWNLOAD.rglob('csa_full_xe_10_v2_review.zip') if hashlib.sha256(p.read_bytes()).hexdigest()==manifest['new_review_zip_sha256']]
    if len(matches)!=1:raise RuntimeError('Exact139 review missing or changed; no latest fallback.')
    review=matches[0]
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    CHAIR=SOURCE/'chair'
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    if hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()!='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':raise ValueError('Changed CHAIR cache')
    run([sys.executable,'-u',SOURCE/'score_saved_quality.py','--review',review,'--baseline',SOURCE/'comparison_baseline4000.json.gz',
         '--manifest',SOURCE/'COMPARISON_INPUTS.json','--chair',CHAIR,'--output',OUTPUT],timeout=2400)
    result=json.loads((OUTPUT/'comparison.json').read_text(encoding='utf-8'))
    complete=result['status']=='SAVED_CAPTION_COMPARISON_COMPLETE'
    if not complete:raise RuntimeError('Incomplete comparison')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,model_run=False,test_used=False)),encoding='utf-8')
    archive=Path(str(OUTPUT)+'_review.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file():z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',archive,flush=True)
    print('COMPLETE' if complete else 'INCOMPLETE; no partial-model ranking.',flush=True)
